# 17-620 Lab 2 — Encoding Classical Data and Quantum Similarity

**80 minutes | PennyLane | Fall 2026**

**Learning objectives**
1. Explain why an encoding map changes the similarity that a quantum circuit computes.
2. Implement angle and amplitude embedding in PennyLane.
3. Compute a kernel matrix using circuit overlap.
4. Contrast exact simulator probabilities with finite-shot estimates and a classical similarity baseline.

Read: *A Practical Guide to Quantum Machine Learning and Quantum Optimization*, Chapter 9. No Qiskit is needed.

**Start:** Run the next cell. Complete each TODO and answer the discussion questions in Markdown cells.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

print("PennyLane", qml.__version__)

## Part 1 — Classical baseline (10 minutes)

Consider four two-feature vectors. Compare their cosine similarity matrix. Pay attention to the difference between a negative similarity and zero similarity.

In [ ]:
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 0.0]])
cosine = cosine_similarity(X)
print("Classical cosine similarity:\n", cosine)
plt.imshow(cosine, vmin=-1, vmax=1)
plt.colorbar(label="cosine similarity")
plt.title("Classical cosine similarity")
plt.show()

**Question 1.** Which pairs are orthogonal? What does a negative cosine similarity mean?

## Part 2 — Angle embedding (20 minutes)

Use `qml.AngleEmbedding` with Y rotations. An angle of π maps |0⟩ to |1⟩ up to global phase. For a two-dimensional vector, use two qubits.

**TODO:** Implement `angle_state` using `qml.AngleEmbedding(features=x, wires=[0,1], rotation="Y")`.

In [ ]:
angle_dev = qml.device("default.qubit", wires=2)

@qml.qnode(angle_dev)
def angle_state(x):
    # TODO: replace 'pass' with angle embedding
    pass
    return qml.probs(wires=[0, 1])

# Try these feature angles (radians) once complete.
angle_inputs = np.array([[0.0, 0.0], [np.pi/2, 0.0],
                         [0.0, np.pi/2], [np.pi/2, np.pi/2]])
# print(angle_state(angle_inputs[0]))
# print(qml.draw(angle_state)(angle_inputs[0]))

**Question 2.** Why are features expressed as angles here? What happens if an input changes by 2π?

## Part 3 — Overlap kernel (20 minutes)

For a feature map U(x)|00⟩ = |φ(x)⟩, the circuit U(x) followed by U(y)† returns |00⟩ with probability

K(x,y) = |⟨φ(y)|φ(x)⟩|².

**TODO:** Fill in the two embedding operations. The adjoint must be applied to the *second* embedding.

In [ ]:
kernel_dev = qml.device("default.qubit", wires=2)

@qml.qnode(kernel_dev)
def angle_overlap(x, y):
    # TODO: embed x with qml.AngleEmbedding
    # TODO: undo embedding y using qml.adjoint(qml.AngleEmbedding)(...)
    return qml.probs(wires=[0, 1])[0]

def make_kernel(points, circuit):
    return np.array([[circuit(x, y) for y in points] for x in points])

# TODO: compute and display K_angle = make_kernel(angle_inputs, angle_overlap)
# Verify K is symmetric, its diagonal is 1, and all entries lie in [0, 1].

**Question 3.** Is this kernel the same as Euclidean distance or cosine similarity? Give an example where its interpretation differs.

## Part 4 — Amplitude embedding (15 minutes)

Amplitude embedding loads a vector into the amplitudes of computational-basis states, after normalization. With two qubits we can represent a four-component vector. Compare normalized vectors using squared inner products.

**TODO:** Implement amplitude overlap with `qml.AmplitudeEmbedding(..., normalize=True)` and its adjoint.

In [ ]:
amp_dev = qml.device("default.qubit", wires=2)

@qml.qnode(amp_dev)
def amplitude_overlap(a, b):
    # TODO: embed a and undo the embedding of b
    return qml.probs(wires=[0, 1])[0]

A = np.array([[1., 0., 0., 0.], [0., 1., 0., 0.],
              [1., 1., 0., 0.], [-1., 0., 0., 0.]])
# TODO: compute K_amp = make_kernel(A, amplitude_overlap)
# Compare to (cosine_similarity(A))**2.

**Question 4.** Why can a sign flip disappear under squared overlap? What do qubit count and state-preparation cost tell us about practical advantage?

## Part 5 — Finite-shot measurement (15 minutes)

The simulator gave exact probabilities. Now estimate an overlap with repeated measurements. This is a circuit-based estimate, not direct access to a hidden state vector.

In [ ]:
shots = 1000
sample_dev = qml.device("default.qubit", wires=2, shots=shots)

@qml.qnode(sample_dev)
def sampled_angle_overlap(x, y):
    # TODO: embed x and apply adjoint embedding for y
    return qml.sample(wires=[0, 1])

# TODO: count all-zero bitstrings and divide by shots
# Compare the estimate with angle_overlap(x,y) for an interesting pair.

**Exit ticket**
1. State exactly what the overlap kernel measures.
2. Identify one reason an amplitude-encoded representation does **not** automatically give a quantum speedup.
3. Explain how a finite number of shots changes the kernel estimate.
4. Propose a fair classical baseline for a future classifier or clustering task.

**Optional extension:** Feed a *validated* precomputed kernel matrix to `sklearn.svm.SVC(kernel="precomputed")` on a labeled dataset; evaluate on held-out samples. Do not claim quantum advantage without measuring state preparation, sampling, and classification costs.